# Phần 5: Đánh giá Model chi tiết

Model tốt nhất (từ Phần 4/6) đã được lưu ở `models/best_model.pkl`.
Notebook này đào sâu hơn RMSLE/MAE đơn thuần: xem residual có pattern
hệ thống không, model yếu ở phân khúc giá nào, và những căn nhà nào bị
dự đoán sai nhiều nhất.

> Logic chi tiết nằm trong `src/evaluate.py`. Notebook này gọi lại các
> hàm đó để trực quan hóa và diễn giải.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from data_loader import load_processed_data
from evaluate import (plot_predicted_vs_actual, plot_residuals,
                       error_by_price_segment, worst_predictions)

df = load_processed_data()
X = df.drop(columns=['price'])
y = np.log1p(df['price'])

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = joblib.load('../models/best_model.pkl')
preds_log = model.predict(X_test)

y_true = np.expm1(y_test)
preds = np.expm1(preds_log)

## 1. Predicted vs Actual

In [ ]:
plot_predicted_vs_actual(y_true, preds)
plt.show()

## 2. Phân tích Residual

**Lưu ý quan trọng:** residual được tính trên **log-scale**, không phải
USD. Nếu tính trên USD, biểu đồ sẽ tự động trông như hình phễu (residual
lớn dần theo giá) đơn giản vì 10% sai số của nhà 2 triệu (200k) luôn lớn
hơn 10% của nhà 300k (30k) về số tuyệt đối — không phản ánh đúng việc
model có thực sự kém hơn ở phân khúc giá cao hay không. Trên log-scale,
residual xấp xỉ sai số phần trăm, so sánh công bằng giữa các phân khúc.

**Điều cần tìm:**
- Residual vs Predicted: lý tưởng là một đám mây ngẫu nhiên quanh 0,
  không có hình dạng cong hay phễu (nếu có → model thiếu 1 pattern nào đó).
- Phân phối residual: lý tưởng giống phân phối chuẩn, tâm ở 0 (nếu lệch
  hẳn sang 1 phía → model có bias hệ thống, ví dụ luôn đoán thấp).

In [ ]:
residuals = plot_residuals(y_test, preds_log)
plt.show()

print(f'Residual trung bình: {residuals.mean():.4f} (gần 0 là tốt, không bias)')
print(f'Residual std: {residuals.std():.4f}')

## 3. Sai số theo phân khúc giá

Chia nhà thành 5 nhóm từ rẻ nhất (Q1) đến đắt nhất (Q5), xem model có
yếu hẳn ở đầu nào không.

In [ ]:
segment_df = error_by_price_segment(y_true, preds)
segment_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(segment_df['price_segment'], segment_df['MdAPE'], color='coral')
axes[0].set_title('Sai số % trung vị theo phân khúc giá')
axes[0].set_ylabel('MdAPE (%)')

axes[1].bar(segment_df['price_segment'], segment_df['MAE'], color='steelblue')
axes[1].set_title('MAE (USD) theo phân khúc giá')
axes[1].set_ylabel('MAE (USD)')
plt.tight_layout()
plt.show()

**Nhận xét:** Sai số % (MdAPE) cao nhất ở Q1 (nhà rẻ nhất) và Q5 (nhà
đắt nhất), thấp nhất ở khoảng giữa (Q2-Q4). Điều này hợp lý:
- Nhà rẻ (Q1): dao động giá tương đối lớn dù chênh lệch USD nhỏ, dễ lệch %.
- Nhà đắt (Q5): ít mẫu hơn, đa dạng hơn (biệt thự, waterfront, nhà cổ...),
  khó khái quát hóa hơn nhà bình dân phổ biến.

`MAE` (USD) tăng dần theo phân khúc giá — đúng như dự đoán ở phần đầu
notebook (10% của nhà đắt luôn là nhiều USD hơn 10% của nhà rẻ).

## 4. Top các dự đoán sai lệch nhiều nhất

Xem cụ thể những căn nhà nào bị đoán sai nhiều nhất và đặc điểm của chúng
— giúp phát hiện model yếu ở loại nhà nào, hoặc nghi ngờ dữ liệu bị lỗi.

In [ ]:
worst_df = worst_predictions(y_true, preds, X_test, top_n=10)
worst_df

**Nhận xét:** Một vài trường hợp tệ nhất là các căn nhà giá rất cao
(1.2 - 2.2 triệu USD) nhưng `sqft_living` lại khá nhỏ (1100 - 2600 sqft)
và không có `waterfront`/`view` cao — nghĩa là giá trị của chúng đến từ
yếu tố mà dataset **không có** (ví dụ: vị trí đắc địa cụ thể trong thành
phố, kiến trúc đặc biệt, đất rộng...). Đây là giới hạn tự nhiên của
dataset chứ không hẳn là lỗi của model — một hướng cải thiện tiềm năng
là bổ sung thêm dữ liệu vị trí chi tiết hơn (tọa độ, khoảng cách trung tâm).